In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/vikisklyarenko/video-drobilka/video.avi


In [2]:
# Устанавливаем roboflow
!pip install -q -U ultralytics roboflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.8/302.8 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 108.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 5.1 MB/s eta 0:00:00


In [14]:
# Импотрируем необходимые бибилиотеки

import json
import os
import shutil
import subprocess
import numpy as np
import yaml
import cv2
from collections import Counter
from pathlib import Path
 
from roboflow import Roboflow
from ultralytics import YOLO

In [5]:
# Настройки

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
ROBOFLOW_API_KEY = secrets.get_secret("ROBOFLOW_API_KEY")
os.environ["KAGGLE_API_TOKEN"] = secrets.get_secret("KAGGLE_API_TOKEN")
 
WORKSPACE = "vikis-workspace"
PROJECT = "sklyarenko_vl"
VERSION = 4 
 
MODEL = "yolov8m.pt"
EPOCHS = 70
IMGSZ = 640
BATCH = 16
 
CLASS_NAMES = ["crusher", "rock"]

CHECKPOINT_DATASET_SLUG = "vikisklyarenko/crusher-checkpoints"
SAVE_EVERY_N_EPOCHS = 5 

os.chdir("/kaggle/working")

In [6]:
# Скачиваем датасет

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download("yolov8")
dataset_path = Path(dataset.location)

print("Dataset:", dataset_path)

with open(dataset_path / "data.yaml", encoding="utf-8") as f:
    data = yaml.safe_load(f)

names = data["names"]
if isinstance(names, dict):
    names = [names[i] for i in sorted(names)]

print("Классы в датасете:", names)

if len(names) != 2:
    raise RuntimeError(f"Ожидалось 2 класса, получено {len(names)}: {names}")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Sklyarenko_VL-4 in yolov8:: 100%|██████████| 35697/35697 [00:03<00:00, 9516.84it/s] 


Dataset: /kaggle/working/Sklyarenko_VL-4
Классы в датасете: ['crusher', 'rock']


In [7]:
# Часть разметки была в автоматичеком режиме, поэтому вместо прямоугольков была наложена маска сегментации. Для приведения разместки к единому формату, 
# нужно переформатировать данные
# Очистка labels: приводим всё к формату YOLO detection 


def convert_label_file(label_file: Path) -> Counter:
    stats = Counter()
    result = []

    with open(label_file, encoding="utf-8") as f:
        lines = f.readlines()

    for line in lines:
        parts = line.strip().split()
        if not parts:
            continue

        try:
            cls = int(float(parts[0]))
        except ValueError:
            stats["bad_class"] += 1
            continue

        if cls not in (0, 1):
            stats["bad_class"] += 1
            continue

        # class x y w h - обычный bbox
        if len(parts) == 5:
            try:
                x, y, w, h = map(float, parts[1:])
            except ValueError:
                stats["bad_coordinates"] += 1
                continue

            bad = w <= 0 or h <= 0 or not all(0 <= v <= 1 for v in (x, y, w, h))
            if bad:
                stats["bad_coordinates"] += 1
                continue

            result.append(f"{cls} {x:.6f} {y:.6f} {w:.6f} {h:.6f}")
            stats["detection"] += 1
            continue

        # полигон: class x1 y1 x2 y2 ... - переводим в обычный bbox
        if len(parts) >= 7 and (len(parts) - 1) % 2 == 0:
            try:
                coords = np.array(parts[1:], dtype=np.float32)
            except ValueError:
                stats["bad_polygon"] += 1
                continue

            xs, ys = coords[0::2], coords[1::2]

            if len(xs) < 3 or xs.min() < 0 or xs.max() > 1 or ys.min() < 0 or ys.max() > 1:
                stats["bad_polygon"] += 1
                continue

            x_min, x_max, y_min, y_max = xs.min(), xs.max(), ys.min(), ys.max()
            w, h = x_max - x_min, y_max - y_min

            if w <= 0 or h <= 0:
                stats["bad_polygon"] += 1
                continue

            xc, yc = (x_min + x_max) / 2, (y_min + y_max) / 2
            result.append(f"{cls} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")
            stats["polygon_to_box"] += 1
            continue

        stats["bad_format"] += 1

    with open(label_file, "w", encoding="utf-8") as f:
        if result:
            f.write("\n".join(result) + "\n")

    return stats


print("\nОчистка labels...")
total_stats = Counter()

for split in ("train", "valid", "test"):
    labels_dir = dataset_path / split / "labels"
    if not labels_dir.exists():
        print(f"{split}: labels отсутствуют")
        continue

    stats = Counter()
    for label_file in labels_dir.glob("*.txt"):
        stats.update(convert_label_file(label_file))

    total_stats.update(stats)
    print(f"{split}: {dict(stats)}")

    cache_file = dataset_path / split / "labels.cache"
    if cache_file.exists():
        cache_file.unlink()


Очистка labels...
train: {'detection': 172236, 'polygon_to_box': 2674, 'bad_coordinates': 19}
valid: {'detection': 16348, 'polygon_to_box': 167}
test: {'detection': 8440, 'polygon_to_box': 206}


In [8]:
# Проверка: в train/valid обязательно должен быть класс rock

def count_classes(split: str) -> Counter:
    labels_dir = dataset_path / split / "labels"
    counter = Counter()
    if not labels_dir.exists():
        return counter

    for label_file in labels_dir.glob("*.txt"):
        with open(label_file, encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    counter[int(parts[0])] += 1

    return counter


print("\nРаспределение классов:")
stats_by_split = {}

for split in ("train", "valid", "test"):
    counter = count_classes(split)
    stats_by_split[split] = counter
    print(f"{split}: crusher={counter.get(0, 0)}, rock={counter.get(1, 0)}")

if stats_by_split["train"].get(1, 0) == 0:
    raise RuntimeError("В train отсутствует класс rock.")
if stats_by_split["valid"].get(1, 0) == 0:
    raise RuntimeError("В valid отсутствует класс rock — проверьте разметку в Roboflow.")



Распределение классов:
train: crusher=15672, rock=159238
valid: crusher=1488, rock=15027
test: crusher=742, rock=7904


In [9]:
# Финальный data.yaml

final_yaml = {
    "path": str(dataset_path),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 2,
    "names": CLASS_NAMES,
}

with open(dataset_path / "data.yaml", "w", encoding="utf-8") as f:
    yaml.dump(final_yaml, f, allow_unicode=True, sort_keys=False)

print("\nФинальный data.yaml:")
print(yaml.dump(final_yaml, allow_unicode=True, sort_keys=False))
print("Датасет готов к обучению.\n")


Финальный data.yaml:
path: /kaggle/working/Sklyarenko_VL-4
train: train/images
val: valid/images
test: test/images
nc: 2
names:
- crusher
- rock

Датасет готов к обучению.



In [ ]:
# Обучение (раскомментируется при первом обучении)

# run_dir = Path("/kaggle/working/runs/crusher_yolov8m")
# local_checkpoint = run_dir / "weights" / "last.pt"
 
# # если локального чекпоинта нет (например, сессия перезапустилась),
# # пробуем скачать последний с Kaggle Dataset
# if not local_checkpoint.exists():
#     download_dir = Path("/kaggle/working/checkpoint_download")
#     subprocess.run(
#         ["kaggle", "datasets", "download", "-d", CHECKPOINT_DATASET_SLUG,
#          "-p", str(download_dir), "--unzip"],
#         check=False,
#     )
 
#     downloaded_checkpoint = download_dir / "last.pt"
#     if downloaded_checkpoint.exists():
#         local_checkpoint.parent.mkdir(parents=True, exist_ok=True)
#         shutil.copy(downloaded_checkpoint, local_checkpoint)
#         print("Чекпоинт скачан с Kaggle Dataset:", local_checkpoint)
 
 
# def upload_checkpoint(trainer):
#     """Раз в SAVE_EVERY_N_EPOCHS эпох заливает last.pt в Kaggle Dataset."""
#     epoch = trainer.epoch + 1
#     if epoch % SAVE_EVERY_N_EPOCHS != 0:
#         return
 
#     upload_dir = Path("/kaggle/working/checkpoint_upload")
#     upload_dir.mkdir(exist_ok=True)
#     shutil.copy(trainer.save_dir / "weights" / "last.pt", upload_dir / "last.pt")
 
#     metadata = {
#         "title": "crusher checkpoints",
#         "id": CHECKPOINT_DATASET_SLUG,
#         "licenses": [{"name": "CC0-1.0"}],
#     }
#     with open(upload_dir / "dataset-metadata.json", "w", encoding="utf-8") as f:
#         json.dump(metadata, f)
 
#     subprocess.run(
#         ["kaggle", "datasets", "version", "-p", str(upload_dir),
#          "-m", f"epoch {epoch}", "--dir-mode", "zip"],
#         check=False,
#     )
#     print(f"Чекпоинт эпохи {epoch} загружен в {CHECKPOINT_DATASET_SLUG}")
 
 
# if local_checkpoint.exists():
#     print("Продолжаю обучение с чекпоинта:", local_checkpoint)
#     model = YOLO(str(local_checkpoint))
#     model.add_callback("on_fit_epoch_end", upload_checkpoint)
#     results = model.train(resume=True)
 
# else:
#     model = YOLO(MODEL)
#     model.add_callback("on_fit_epoch_end", upload_checkpoint)
 
#     results = model.train(
#         data=str(dataset_path / "data.yaml"),
#         epochs=EPOCHS,
#         imgsz=IMGSZ,
#         batch=BATCH,
#         patience=20,  # остановиться, если mAP не растёт 20 эпох подряд
#         project="/kaggle/working/runs",
#         name="crusher_yolov8m",
#     )
 
# best_weights = Path(results.save_dir) / "weights" / "best.pt"
# print("\nЛучшие веса сохранены здесь:", best_weights)
# print("Скачать их можно во вкладке Output справа от notebook.")

In [11]:
# from pathlib import Path
# import shutil
# import subprocess
# import os
 

# Скачивает last.pt (и best.pt, если есть) из бэкапа на Kaggle,без запуска самого обучения. 
# Нужно запускать перед make_best_from_last.py, если сессия свежая и в /kaggle/working ничего нет.


# токен Kaggle API (тот же, что использовался при обучении)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["KAGGLE_API_TOKEN"] = UserSecretsClient().get_secret("KAGGLE_API_TOKEN")
except Exception:
    pass
 
CHECKPOINT_DATASET_SLUG = "vikisklyarenko/crusher-checkpoints"
 
run_dir = Path("/kaggle/working/runs/crusher_yolov8m")
weights_dir = run_dir / "weights"
weights_dir.mkdir(parents=True, exist_ok=True)
 
download_dir = Path("/kaggle/working/checkpoint_download")
 
subprocess.run(
    ["kaggle", "datasets", "download", "-d", CHECKPOINT_DATASET_SLUG,
     "-p", str(download_dir), "--unzip"],
    check=True,
)
 
for filename in ("last.pt", "best.pt"):
    src = download_dir / filename
    if src.exists():
        shutil.copy(src, weights_dir / filename)
        print("Скачано:", weights_dir / filename)
    else:
        print("В бэкапе нет файла:", filename)

Dataset URL: https://www.kaggle.com/datasets/vikisklyarenko/crusher-checkpoints
License(s): unknown


100%|██████████| 135M/135M [00:01<00:00, 124MB/s]  



Скачано: /kaggle/working/runs/crusher_yolov8m/weights/last.pt
В бэкапе нет файла: best.pt


In [12]:
# from ultralytics import YOLO
# from pathlib import Path
# import shutil
 
# путь к весам после 60 эпох (сессия перезапустилась, но last.pt сохранился)
run_dir = Path("/kaggle/working/runs/crusher_yolov8m")
last_checkpoint = run_dir / "weights" / "last.pt"
best_checkpoint = run_dir / "weights" / "best.pt"
 
# путь к тому же датасету, на котором обучались (нужен для валидации)
DATA_YAML = "/kaggle/working/Sklyarenko_VL-4/data.yaml"
 

# last.pt после полностью пройденных 60 эпох — это готовая обученная модель. Файла best.pt отдельно нет (потерялся при
# перезапуске сессии), поэтому просто используем last.pt как  финальную модель и копируем его в best.pt для удобства.


 
shutil.copy(last_checkpoint, best_checkpoint)
print("Скопировано:", last_checkpoint, "->", best_checkpoint)
 
# загружаем модель и считаем метрики на валидационной выборке 

model = YOLO(str(best_checkpoint))
 
metrics = model.val(data=DATA_YAML)
 
print("\n================================")
print("МЕТРИКИ МОДЕЛИ (60 эпох)")
print("================================")
print("Precision:", round(metrics.box.mp, 4))
print("Recall:", round(metrics.box.mr, 4))
print("mAP50:", round(metrics.box.map50, 4))
print("mAP50-95:", round(metrics.box.map, 4))

Скопировано: /kaggle/working/runs/crusher_yolov8m/weights/last.pt -> /kaggle/working/runs/crusher_yolov8m/weights/best.pt
Ultralytics 8.4.149 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 93 layers, 25,840,918 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1375.4±440.7 MB/s, size: 42.5 KB)
val: Scanning /kaggle/working/Sklyarenko_VL-4/valid/labels... 1481 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1481/1481 1.1Kit/s 1.3s0.0s
val: /kaggle/working/Sklyarenko_VL-4/valid/images/frame_4194_jpg.rf.d7346c7be88bcae4e4927bd2f5f5b6f9.jpg: 1 duplicate labels removed
val: /kaggle/working/Sklyarenko_VL-4/valid/images/frame_4990_jpg.rf.739e5b39d5f7a8497ed12bc37b436788.jpg: 1 duplicate labels removed
val: /kaggle/working/Sklyarenko_VL-4/valid/images/frame_5005_jpg.rf.ed5f51164a0d47c46b23cc5c70d3d2f9.jpg: 1 duplicate labels removed
val: /kaggle/working/Sklyarenko_VL-4/valid/images/frame_5026_jpg.rf.03c3bbdd99643b

In [15]:
# ПРОГРАММА: обработка видео с дробилкой (модель YOLOv8m с Kaggle)

# Что делает эта программа:
# 1. Открывает видео с камеры
# 2. Кадр за кадром прогоняет его через модель YOLOv8m
# 3. Модель находит на кадре куски породы (класс "rock")
# 4. Программа рисует найденные куски зелёными рамками
# 5. Считает, сколько места (в процентах) порода занимает внутри рабочей зоны дробилки (ROI)
# 6. Сохраняет получившееся видео с рамками и подписями

# Подключаем нужные библиотеки (если мы не сделали до это)
 
# from ultralytics import YOLO   # библиотека с моделью YOLO
# import cv2                     # библиотека для работы с видео и картинками
# import numpy as np             # библиотека для работы с числами и массивами
 
# Настройки — здесь всё, что можно менять под своё видео
 
# путь к обученной модели (файл best.pt после обучения на Kaggle)
MODEL_PATH = "/kaggle/working/runs/crusher_yolov8m/weights/best.pt"
 
# путь к видео, которое нужно обработать
VIDEO_PATH = "/kaggle/input/datasets/vikisklyarenko/video-drobilka/video.avi"
 
# путь, куда сохранить готовое видео с рамками
OUTPUT_PATH = "result_yolov8m.mp4"
 
# номер класса "порода" в модели (0 - crusher, 1 - rock)
ROCK_CLASS_ID = 1
 
# порог уверенности модели: объекты с меньшей уверенностью в расчёт не берём
CONFIDENCE_THRESHOLD = 0.25
 
# координаты рабочей зоны дробилки (ROI) — то есть той области, относительно которой мы считаем процент заполнения.
# Координаты подобраны по кадрам с камеры (1280x720), камера неподвижна, поэтому одни и те же точки подходят для всего видео.
roi_points = np.array([
    [285, 237],   # верхний левый угол
    [626, 179],   # верхний правый угол
    [802, 719],   # нижний правый угол
    [333, 719]    # нижний левый угол
], dtype=np.int32)
 
# Загружаем модель

model = YOLO(MODEL_PATH)
print("Модель загружена:", MODEL_PATH)
 
# Открываем видео и готовим файл для записи результата
 
cap = cv2.VideoCapture(VIDEO_PATH)
 
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
 
print("Размер кадра:", frame_width, "x", frame_height)
print("FPS видео:", fps)
 
video_writer = cv2.VideoWriter(
    OUTPUT_PATH,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (frame_width, frame_height)
)
 
 
# Считаем площадь ROI один раз до начала обработки
 
roi_mask = np.zeros((frame_height, frame_width), dtype=np.uint8)
cv2.fillPoly(roi_mask, [roi_points], 255)
roi_area = cv2.countNonZero(roi_mask)
 
print("Площадь ROI (в пикселях):", roi_area)

 
# Главный цикл — обрабатываем видео кадр за кадром

frame_number = 0
 
while cap.isOpened():
 
    success, frame = cap.read()
 
    if not success:
        break
 
    frame_number += 1
 
    cv2.polylines(frame, [roi_points], True, (0, 0, 255), 2)
 
    detections = model(frame, conf=CONFIDENCE_THRESHOLD, verbose=False)[0]
 
    rock_count = 0
    rock_area = 0
 
    for box in detections.boxes:
 
        class_id = int(box.cls[0])
        confidence = float(box.conf[0])
 
        if class_id != ROCK_CLASS_ID:
            continue
 
        x1, y1, x2, y2 = map(int, box.xyxy[0])
 
        center_x = (x1 + x2) // 2
        center_y = (y1 + y2) // 2
 
        is_inside_roi = cv2.pointPolygonTest(roi_points, (center_x, center_y), False)
 
        if is_inside_roi < 0:
            continue
 
        object_area = (x2 - x1) * (y2 - y1)
        object_fraction = object_area / roi_area
 
        rock_count += 1
        rock_area += object_area
 
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 3)
 
        label = f"rock: {confidence * 100:.1f}%"
        cv2.putText(frame, label, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
 
        cv2.circle(frame, (center_x, center_y), 4, (0, 0, 255), -1)
 
        cv2.putText(frame, f"{center_x}:{center_y}", (center_x + 8, center_y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
 
        cv2.putText(frame, f"{object_area}:{object_fraction:.6f}", (center_x + 8, center_y + 18),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
 
    fill_fraction = rock_area / roi_area
 
    cv2.putText(frame, f"Frame {frame_number}", (10, 25),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
 
    cv2.putText(frame, f"Rock count {rock_count}", (10, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
 
    cv2.putText(frame, f"Sum frame area percent {fill_fraction:.6f}", (10, 75),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
 
    cv2.putText(frame, f"Sum frame area {rock_area}", (10, 100),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
 
    video_writer.write(frame)
 
 
# Закрываем видео-файлы

cap.release()
video_writer.release()
 
print("Готово! Обработано кадров:", frame_number)
print("Результат сохранён в:", OUTPUT_PATH)

Модель загружена: /kaggle/working/runs/crusher_yolov8m/weights/best.pt
Размер кадра: 1280 x 720
FPS видео: 11.0278991008991
Площадь ROI (в пикселях): 211120
Готово! Обработано кадров: 27900
Результат сохранён в: result_yolov8m.mp4


In [16]:
# скачиваем итоговое видео
from IPython.display import FileLink
FileLink("result_yolov8m.mp4")

/kaggle/working/result_yolov8m.mp4